# Exploratory Data Analysis (EDA)

### Libraries

In [1]:
import pandas as pd
import os
import scipy.stats as stats

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

os.chdir("../1 - EXPLORATORY DATA ANALYSIS")

### Import Data

In [2]:
raw_df = pd.read_csv('exploratory_data_analysis.csv')
raw_df.head(10)
#print(raw_df.shape)

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0000,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0000,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0000,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0000,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0000,0,1,1
5,2011-01-01 05:00:00,1,0,0,2,9.84,12.880,75,6.0032,0,1,1
6,2011-01-01 06:00:00,1,0,0,1,9.02,13.635,80,0.0000,2,0,2
7,2011-01-01 07:00:00,1,0,0,1,8.20,12.880,86,0.0000,1,2,3
8,2011-01-01 08:00:00,1,0,0,1,9.84,14.395,75,0.0000,1,7,8
9,2011-01-01 09:00:00,1,0,0,1,13.12,17.425,76,0.0000,8,6,14


General initial thoughts/Assumptions:

1) Holiday and Workingday are mutually exlusive. It can be assumed that is it is a holiday then it cannot be a working day. However, not all non-working days are holidays; there are weekends. 
2) Count is the sum of casual and registered users. 

### Data Engineering

In [3]:
df_eng = raw_df.copy()
df_eng['datetime'] = pd.to_datetime(df_eng['datetime'])

cols = ["season", "holiday", "workingday"]

for col in cols:
    df_eng[col] = raw_df[col].astype('object')

In [4]:
df_eng.describe(include='all')

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
count,10886,10886.0,10886.0,10886.0,10886.000000,10886.00000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000,10886.000000
unique,NaN,4.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,4.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,2734.0,10575.0,7412.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2011-12-27 05:56:22.399411968,NaN,NaN,NaN,1.418427,20.23086,23.655084,61.886460,12.799395,36.021955,155.552177,191.574132
min,2011-01-01 00:00:00,NaN,NaN,NaN,1.000000,0.82000,0.760000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,2011-07-02 07:15:00,NaN,NaN,NaN,1.000000,13.94000,16.665000,47.000000,7.001500,4.000000,36.000000,42.000000
50%,2012-01-01 20:30:00,NaN,NaN,NaN,1.000000,20.50000,24.240000,62.000000,12.998000,17.000000,118.000000,145.000000
75%,2012-07-01 12:45:00,NaN,NaN,NaN,2.000000,26.24000,31.060000,77.000000,16.997900,49.000000,222.000000,284.000000
max,2012-12-19 23:00:00,NaN,NaN,NaN,4.000000,41.00000,45.455000,100.000000,56.996900,367.000000,886.000000,977.000000


In [ ]:
df_eng
#Filter rows where holiday = 1 and workingday = 1, i.e are there working holidays?
filtered_df = df_eng[(df_eng['holiday'] == 1) & (df_eng['workingday'] == 1)]
print(filtered_df)

Empty DataFrame
Columns: [datetime, season, holiday, workingday, weather, temp, atemp, humidity, windspeed, casual, registered, count]
Index: []


In [6]:
df_eda = df_eng.copy()

### Correlation/Statistical Relationship

In [ ]:
#Create a subgroup of DataFrame to test correlation using Pearson correlation for continuous variables

df_corr = df_eda[['temp', 'humidity', 'windspeed', 'casual', 'registered']]
fig_corr_1 = px.imshow(df_corr.corr('pearson'), text_auto=True, aspect="auto", color_continuous_scale='RdBu_r', title='Correlation Matrix')
fig_corr_1.update_layout(margin=dict(l=20, r=20, t=50, b=20))
fig_corr_1.show()


#### Notes: 
Not much difference between temp and atemp, so we can drop one of them. Humidity and Windspeed have low correlation with Casual and Registered. However, temp has a strong correlation with Casual and Registered.

*Initial Hypothesis:* 
1. As temperature increases, more people are likely to rent cars, whether they are casual or registered users.
2. Humidity seems negatively correlated, this could be because high humidity often leads to discomfort, which might deter people from outdoor activities, or it's simply rainy. Need more data to confirm. 

##### Correlation Ratio for Categorical Variables that cannot be measured by Pearson correlation

Sources: 
* https://en.wikipedia.org/wiki/Correlation_ratio 
* https://www.statisticssolutions.com/free-resources/directory-of-statistical-analyses/correlation-ratio/ 

In [29]:
import numpy as np

def correlation_ratio(categories, values):
    categories = np.array(categories)
    values = np.array(values)
    
    ssw = 0
    ssb = 0
    for category in set(categories):
        subgroup = values[np.where(categories == category)[0]]
        ssw += sum((subgroup-np.mean(subgroup))**2)
        ssb += len(subgroup)*(np.mean(subgroup)-np.mean(values))**2

    return (ssb / (ssb + ssw))**.5

# Calculate correlation ratio for categorical variables
categorical_cols = ['season', 'holiday', 'workingday', 'weather']
correlation_ratios = {}
for col in categorical_cols:
    correlation_ratios[col] = correlation_ratio(df_eda[col], df_eda['count'])
print("Correlation Ratios with all Users:")
for col, ratio in correlation_ratios.items():
    print(f"{col}: {ratio:.4f}")
    

Correlation Ratios with all Users:
season: 0.2476
holiday: 0.0054
workingday: 0.0116
weather: 0.1332


In [28]:
import numpy as np

def correlation_ratio(categories, values):
    categories = np.array(categories)
    values = np.array(values)
    
    ssw = 0
    ssb = 0
    for category in set(categories):
        subgroup = values[np.where(categories == category)[0]]
        ssw += sum((subgroup-np.mean(subgroup))**2)
        ssb += len(subgroup)*(np.mean(subgroup)-np.mean(values))**2

    return (ssb / (ssb + ssw))**.5

# Calculate correlation ratio for categorical variables
categorical_cols = ['season', 'holiday', 'workingday', 'weather']
correlation_ratios = {}
for col in categorical_cols:
    correlation_ratios[col] = correlation_ratio(df_eda[col], df_eda['registered'])
print("Correlation Ratios with Registered Users:")
for col, ratio in correlation_ratios.items():
    print(f"{col}: {ratio:.4f}")
    

Correlation Ratios with Registered Users:
season: 0.2104
holiday: 0.0210
workingday: 0.1195
weather: 0.1154


In [30]:
import numpy as np

def correlation_ratio(categories, values):
    categories = np.array(categories)
    values = np.array(values)
    
    ssw = 0
    ssb = 0
    for category in set(categories):
        subgroup = values[np.where(categories == category)[0]]
        ssw += sum((subgroup-np.mean(subgroup))**2)
        ssb += len(subgroup)*(np.mean(subgroup)-np.mean(values))**2

    return (ssb / (ssb + ssw))**.5

# Calculate correlation ratio for categorical variables
categorical_cols = ['season', 'holiday', 'workingday', 'weather']
correlation_ratios = {}
for col in categorical_cols:
    correlation_ratios[col] = correlation_ratio(df_eda[col], df_eda['casual'])
print("Correlation Ratios with Casual Users:")
for col, ratio in correlation_ratios.items():
    print(f"{col}: {ratio:.4f}")
    

Correlation Ratios with Casual Users:
season: 0.2946
holiday: 0.0438
workingday: 0.3191
weather: 0.1366


To verify the nonsignificance of correlation ratios, point-biserial correlation is calculated for binary categorical variables 'holiday' and 'workingday'.

##### Calculate point-biserial correlation

Source:
* https://www.statology.org/point-biserial-correlation-python/

In [9]:
x=df_eda['holiday']
y=df_eda['count']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(-0.005392984477774103), pvalue=np.float64(0.5736923883271366))

Not statistically sigificant as shown by p-value > 0.05.

In [10]:
x=df_eda['workingday']
y=df_eda['count']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(0.01159386609157469), pvalue=np.float64(0.2264480422636019))

In [31]:
x=df_eda['workingday']
y=df_eda['casual']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(-0.31911096340430434), pvalue=np.float64(3.561967423504164e-256))

In [32]:
x=df_eda['holiday']
y=df_eda['casual']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(0.04379892867534405), pvalue=np.float64(4.843060024101483e-06))

In [34]:
x=df_eda['holiday']
y=df_eda['registered']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(-0.02095567293532982), pvalue=np.float64(0.028784891923612672))

In [33]:
x=df_eda['workingday']
y=df_eda['registered']

#Calculate point-biserial correlation
stats.pointbiserialr(x, y)

SignificanceResult(statistic=np.float64(0.11945985076843983), pvalue=np.float64(6.806493719889584e-36))

Not statistically sigificant as shown by p-value > 0.05.

The calculation of the correlation ratio and point-biserial correlation have both demonstrated that working days and holidays do not have a strong statistical significance with the target variable 'count'.

#### One-hot encoding for season, holiday, workingday, weather


In [11]:
df_dummies = pd.get_dummies(df_eda, columns=['season', 'holiday', 'workingday', 'weather'], drop_first=False, dtype=int)
df_dummies.head(2)

,datetime,temp,atemp,humidity,windspeed,casual,registered,count,season_1,season_2,season_3,season_4,holiday_0,holiday_1,workingday_0,workingday_1,weather_1,weather_2,weather_3,weather_4
0,2011-01-01 00:00:00,9.84,14.395,81,0.0,3,13,16,1,0,0,0,1,0,1,0,1,0,0,0
1,2011-01-01 01:00:00,9.02,13.635,80,0.0,8,32,40,1,0,0,0,1,0,1,0,1,0,0,0


#### Metrics over time

In [12]:
#Create new feature 'year_month' for monthly analysis
df_eda['year_month'] = df_eda['datetime'].dt.to_period('M').dt.to_timestamp()

* There is no need to break down by season as this information is already captured in the month variable.

In [13]:
#Aggregate data by month and workingday

df_monthly_workingday = df_eda.groupby(['year_month', 'workingday'], as_index=False)[['count', 'casual', 'registered']].sum()
workingdays = df_monthly_workingday['workingday'].unique()

traces = []
for y_col in ['casual', 'registered', 'count']:
    for wd in workingdays:
        trace = go.Bar(
            x=df_monthly_workingday[df_monthly_workingday['workingday'] == wd]['year_month'],
            y=df_monthly_workingday[df_monthly_workingday['workingday'] == wd][y_col],
            name=f"{y_col} - workingday {wd}",
            visible=True if y_col=='casual' else False
        )
        traces.append(trace)

buttons = []
for i, y_col in enumerate(['casual', 'registered', 'total']):
    visibility = []
    for j in range(len(traces)):
        visibility.append(i == j // len(workingdays))
    buttons.append(dict(label=y_col,
                        method="update",
                        args=[{"visible": visibility},
                              {"title": f"{y_col} by Month"}]))

fig = go.Figure(data=traces)
fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons)],
    title="Casual Users by Month",
    barmode='relative',
)
fig.show()

In [14]:
#Aggregate data by month and holiday

df_monthly_holiday = df_eda.groupby(['year_month', 'holiday'], as_index=False)[['count', 'casual', 'registered']].sum()
holidays = df_monthly_holiday['holiday'].unique()

traces = []
for y_col in ['casual', 'registered', 'count']:
    for wd in holidays:
        trace = go.Bar(
            x=df_monthly_holiday[df_monthly_holiday['holiday'] == wd]['year_month'],
            y=df_monthly_holiday[df_monthly_holiday['holiday'] == wd][y_col],
            name=f"{y_col} - holiday {wd}",
            visible=True if y_col=='casual' else False
        )
        traces.append(trace)

buttons = []
for i, y_col in enumerate(['casual', 'registered', 'total']):
    visibility = []
    for j in range(len(traces)):
        visibility.append(i == j // len(holidays))
    buttons.append(dict(label=y_col,
                        method="update",
                        args=[{"visible": visibility},
                              {"title": f"{y_col} by Month"}]))

fig = go.Figure(data=traces)
fig.update_layout(
    updatemenus=[dict(active=0, buttons=buttons)],
    title="Casual Users by Month",
    barmode='relative',
)
fig.show()

In [15]:
#Create subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,
                    subplot_titles=("Total Users", "Casual Users", "Registered Users"))

df_monthly_line = df_eda.groupby(['year_month', 'workingday'], as_index=False)[['count', 'casual', 'registered']].sum()

#Total Users
fig_total = px.line(df_monthly_workingday, x="year_month", y="count", color="workingday")
for trace in fig_total.data:
    fig.add_trace(trace, row=1, col=1)

#Casual Users
fig_casual = px.line(df_monthly_workingday, x="year_month", y="casual", color="workingday")
for trace in fig_casual.data:
    fig.add_trace(trace, row=2, col=1)

#Registered Users
fig_registered = px.line(df_monthly_workingday, x="year_month", y="registered", color="workingday")
for trace in fig_registered.data:
    fig.add_trace(trace, row=3, col=1)

for i in range(1, 4):
    fig.update_xaxes(title_text="Month", row=i, col=1)

fig.update_yaxes(title_text="Total Users", row=1, col=1)
fig.update_yaxes(title_text="Casual Users", row=2, col=1)
fig.update_yaxes(title_text="Registered Users", row=3, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Users by Month and Working Day",
    legend=dict(
        orientation="v",
        x=1.02,   
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show()

In [16]:
#Create subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,
                    subplot_titles=("Total Users", "Casual Users", "Registered Users"))

#Total Users
fig_total = px.line(df_monthly_holiday, x="year_month", y="count", color="holiday")
for trace in fig_total.data:
    fig.add_trace(trace, row=1, col=1)

#Casual Users
fig_casual = px.line(df_monthly_holiday, x="year_month", y="casual", color="holiday")
for trace in fig_casual.data:
    fig.add_trace(trace, row=2, col=1)

#Registered Users
fig_registered = px.line(df_monthly_holiday, x="year_month", y="registered", color="holiday")
for trace in fig_registered.data:
    fig.add_trace(trace, row=3, col=1)

for i in range(1, 4):
    fig.update_xaxes(title_text="Month", row=i, col=1)

fig.update_yaxes(title_text="Total Users", row=1, col=1)
fig.update_yaxes(title_text="Casual Users", row=2, col=1)
fig.update_yaxes(title_text="Registered Users", row=3, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Users by Month and Holiday",
    legend=dict(
        orientation="v",
        x=1.02,   
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show()

#### Trends by the hour of the day

In [ ]:
#Create new feature 'hour' for monthly analysis
df_eda['hour'] = df_eda['datetime'].dt.hour

In [18]:
#Create subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,
                    subplot_titles=("Total Users", "Casual Users", "Registered Users"))

df_hourly_line = df_eda.groupby(['hour', 'workingday'], as_index=False)[['count', 'casual', 'registered']].mean()

#Total Users
fig_total = px.line(df_hourly_line, x="hour", y="count", color="workingday")
for trace in fig_total.data:
    fig.add_trace(trace, row=1, col=1)

#Casual Users
fig_casual = px.line(df_hourly_line, x="hour", y="casual", color="workingday")
for trace in fig_casual.data:
    fig.add_trace(trace, row=2, col=1)

#Registered Users
fig_registered = px.line(df_hourly_line, x="hour", y="registered", color="workingday")
for trace in fig_registered.data:
    fig.add_trace(trace, row=3, col=1)

hours = [f"{h:02d}:00" for h in range(24)]

for i in range(1, 4):
    fig.update_xaxes(
        title_text="Hour", 
        row=i, 
        col=1,
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=hours
        )

fig.update_yaxes(title_text="Total Users", row=1, col=1)
fig.update_yaxes(title_text="Casual Users", row=2, col=1)
fig.update_yaxes(title_text="Registered Users", row=3, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Users by Hour and Working Day",
    legend=dict(
        orientation="v",
        x=1.02,   
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show()

In [19]:
#Create subplots
fig = make_subplots(rows=3, cols=1, shared_xaxes=False,
                    subplot_titles=("Total Users", "Casual Users", "Registered Users"))

df_hourly_line_holiday = df_eda.groupby(['hour', 'holiday'], as_index=False)[['count', 'casual', 'registered']].mean()

#Total Users
fig_total = px.line(df_hourly_line_holiday, x="hour", y="count", color="holiday")
for trace in fig_total.data:
    fig.add_trace(trace, row=1, col=1)

#Casual Users
fig_casual = px.line(df_hourly_line_holiday, x="hour", y="casual", color="holiday")
for trace in fig_casual.data:
    fig.add_trace(trace, row=2, col=1)

#Registered Users
fig_registered = px.line(df_hourly_line_holiday, x="hour", y="registered", color="holiday")
for trace in fig_registered.data:
    fig.add_trace(trace, row=3, col=1)

hours = [f"{h:02d}:00" for h in range(24)]

for i in range(1, 4):
    fig.update_xaxes(
        title_text="Hour", 
        row=i, 
        col=1,
        tickmode="array",
        tickvals=list(range(24)),
        ticktext=hours
        )

fig.update_yaxes(title_text="Total Users", row=1, col=1)
fig.update_yaxes(title_text="Casual Users", row=2, col=1)
fig.update_yaxes(title_text="Registered Users", row=3, col=1)

fig.update_layout(
    height=900,
    width=950,
    title_text="Users by Hour and Holiday",
    legend=dict(
        orientation="v",
        x=1.02,   
        y=1,
        xanchor="left",
        yanchor="top"
    )
)

fig.show()

Time of day seems to be significant as we can see peaks and troughs in user activity, especially during morning and evening hours. This pattern is likely influenced by commuting behaviors, with higher usage during typical work commute times. Additionally, the distinction between working days and holidays further emphasizes the impact of daily routines on user activity.

### Interative Scatter Plot for Temperature and Humidity

In [20]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]

fig = go.Figure()

for y_col in y_options:
    for season, label in season_labels.items():
        df_season = df_eda[df_eda["season"] == season]
        fig.add_trace(
            go.Scatter(
                x=df_season["atemp"],
                y=df_season[y_col],
                mode="markers",
                marker=dict(
                    size=df_season["count"],
                    color=df_season["atemp"],
                    colorscale="oranges",
                    showscale=False,
                    sizemode="area",
                    sizeref=2.*max(df_season["count"])/40**2,
                    sizemin=4
                ),
                name=f"{label} - {y_col}",
                visible=True if (season==1 and y_col=="count") else False,
            )
        )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="oranges",
            cmin=raw_df["atemp"].min(),
            cmax=raw_df["atemp"].max(),
            colorbar=dict(
                title="Temperature (°C)",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Temperature (°C)")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for i, (season, label) in enumerate(season_labels.items()):
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            visibility.append(s == season and y_col == "count")  
    visibility.append(True)  
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count ({label})"}]
    ))
#Dropdown 2: Customer (y-axis)
y_buttons = []
for i, y_col in enumerate(y_options):
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            visibility.append(y_opt == y_col and s == 1)  
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Temperature and {y_col.capitalize()} (Spring)"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        )
    ],
    title="Temperature and Count (Spring)",
    height=600
)

fig.show()

In [21]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]

fig = go.Figure()

for y_col in y_options:
    for season, label in season_labels.items():
        df_season = df_eda[df_eda["season"] == season]
        fig.add_trace(
            go.Scatter(
                x=df_season["humidity"],
                y=df_season[y_col],
                mode="markers",
                marker=dict(
                    size=df_season["count"],
                    color=df_season["humidity"],
                    colorscale="blues",
                    showscale=False,
                    sizemode="area",
                    sizeref=2.*max(df_season["count"])/40**2,
                    sizemin=4
                ),
                name=f"{label} - {y_col}",
                visible=True if (season==1 and y_col=="count") else False,
            )
        )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="blues",
            cmin=raw_df["humidity"].min(),
            cmax=raw_df["humidity"].max(),
            colorbar=dict(
                title="Humidity",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Humidity")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for i, (season, label) in enumerate(season_labels.items()):
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            visibility.append(s == season and y_col == "count")  
    visibility.append(True)  
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count ({label})"}]
    ))
#Dropdown 2: Customer (y-axis)
y_buttons = []
for i, y_col in enumerate(y_options):
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            visibility.append(y_opt == y_col and s == 1)  
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Humidity and {y_col.capitalize()} (Spring)"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        )
    ],
    title="Humidity and Count (Spring)",
    height=600
)

fig.show()

In [22]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]
workingday_labels = {0: "Weekend", 1: "Weekday"}

fig = go.Figure()

for y_col in y_options:
    for season, label in season_labels.items():
        for wd, wd_label in workingday_labels.items():
            df_season = df_eda[(df_eda["season"] == season) & (df_eda["workingday"] == wd)]
            fig.add_trace(
                go.Scatter(
                    x=df_season["atemp"],
                    y=df_season[y_col],
                    mode="markers",
                    marker=dict(
                        size=df_season["count"],
                        color=df_season["atemp"],
                        colorscale="oranges",
                        showscale=False,
                        sizemode="area",
                        sizeref=2.*max(df_season["count"])/40**2,
                        sizemin=4
                    ),
                    name=f"{label} - {y_col} - {wd_label}",
                    visible=True if (season==1 and y_col=="count" and wd==1) else False,
                )
            )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="oranges",
            cmin=raw_df["atemp"].min(),
            cmax=raw_df["atemp"].max(),
            colorbar=dict(
                title="Temperature (°C)",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Temperature (°C)")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for season, label in season_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                visibility.append(s == season and y_col == "count" and wd == 1)
    visibility.append(True)  # for dummy trace
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count ({label}, Weekday)"}]
    ))

#Dropdown 2: Customer type (y-axis variable)
y_buttons = []
for y_col in y_options:
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                visibility.append(y_opt == y_col and s == 1 and wd == 1)
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Temperature and {y_col.capitalize()} (Spring, Weekday)"}]
    ))

#Dropdown 3: Working day
workingday_buttons = []
for wd, wd_label in workingday_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for w in workingday_labels:
                visibility.append(w == wd and s == 1 and y_col == "count")
    visibility.append(True)  
    workingday_buttons.append(dict(
        label=wd_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count (Spring, {wd_label})"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.65,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=workingday_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=1
        )
    ],
    title="Temperature and Count (Spring, Weekday)",
    height=600
)

fig.show()


In [23]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]
workingday_labels = {0: "Weekend", 1: "Weekday"}

fig = go.Figure()

for y_col in y_options:
    for season, label in season_labels.items():
        for wd, wd_label in workingday_labels.items():
            df_season = df_eda[(df_eda["season"] == season) & (df_eda["workingday"] == wd)]
            fig.add_trace(
                go.Scatter(
                    x=df_season["humidity"],
                    y=df_season[y_col],
                    mode="markers",
                    marker=dict(
                        size=df_season["count"],
                        color=df_season["humidity"],
                        colorscale="blues",
                        showscale=False,
                        sizemode="area",
                        sizeref=2.*max(df_season["count"])/40**2,
                        sizemin=4
                    ),
                    name=f"{label} - {y_col} - {wd_label}",
                    visible=True if (season==1 and y_col=="count" and wd==1) else False,
                )
            )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="blues",
            cmin=raw_df["humidity"].min(),
            cmax=raw_df["humidity"].max(),
            colorbar=dict(
                title="Humidity",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Humidity")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for season, label in season_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                visibility.append(s == season and y_col == "count" and wd == 1)
    visibility.append(True) 
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count ({label}, Weekday)"}]
    ))

#Dropdown 2: Customer type (y-axis variable)
y_buttons = []
for y_col in y_options:
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                visibility.append(y_opt == y_col and s == 1 and wd == 1)
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Humidity and {y_col.capitalize()} (Spring, Weekday)"}]
    ))

#Dropdown 3: Working day
workingday_buttons = []
for wd, wd_label in workingday_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for w in workingday_labels:
                visibility.append(w == wd and s == 1 and y_col == "count")
    visibility.append(True)  
    workingday_buttons.append(dict(
        label=wd_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count (Spring, {wd_label})"}]
    ))

#Add dropdowns to layout
fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.65,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=workingday_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=1
        )
    ],
    title="Humidity and Count (Spring, Weekday)",
    height=600
)

fig.show()


### Full Interactive EDA Chart

In [ ]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]
workingday_labels = {0: "Weekend", 1: "Weekday"}
#weather_labels = {
    #1: "Clear, Few clouds,<br>Partly Cloudy", 
    #2: "Mist + Cloudy,<br>Mist + Few clouds,<br>Mist", 
    #3: "Light Snow,<br>Light Rain + Thunderstorm +<br>Scattered Clouds,<br>Light Rain + Scattered Clouds", 
    #4: "Heavy Rain + Ice Pellets +<br>Thunderstorm + Mist,<br>Snow + Fog"}

weather_labels = {
    1: "Clear",
    2: "Mist",
    3: "Light Precip",
    4: "Heavy Precip"
}

fig = go.Figure()

global_sizeref = 2. * df_eda["count"].max() / 40**2

for y_col in y_options:
    for season, label in season_labels.items():
        for wd, wd_label in workingday_labels.items():
            for wthr, wthr_label in weather_labels.items():
                df_season = df_eda[
                    (df_eda["season"] == season) &
                    (df_eda["workingday"] == wd) &
                    (df_eda["weather"] == wthr)
                ]
                
                if df_season.empty:
                    continue 
                
                fig.add_trace(
                    go.Scatter(
                        x=df_season["atemp"],
                        y=df_season[y_col],
                        mode="markers",
                        marker=dict(
                            size=df_season["count"],
                            color=df_season["atemp"],
                            colorscale="oranges",
                            showscale=False,
                            sizemode="area",
                            sizeref=global_sizeref, 
                            sizemin=4
                        ),
                        name=f"{label} - {y_col} - {wd_label} - {wthr_label}",
                        visible=True if (season==1 and y_col=="count" and wd==1 and wthr==1) else False,
                    )
                )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="oranges",
            cmin=raw_df["atemp"].min(),
            cmax=raw_df["atemp"].max(),
            colorbar=dict(
                title="Temperature (°C)",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Temperature (°C)")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for season, label in season_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(s == season and y_col == "count" and wd == 1 and wthr == 1)
    visibility.append(True)  
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count ({label}, Weekday, Clear)"}]
    ))

#Dropdown 2: Customer type
y_buttons = []
for y_col in y_options:
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(y_opt == y_col and s == 1 and wd == 1 and wthr == 1)
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Temperature and {y_col.capitalize()} (Spring, Weekday, Clear)"}]
    ))

#Dropdown 3: Working day
workingday_buttons = []
for wd, wd_label in workingday_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for w in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(w == wd and s == 1 and y_col == "count" and wthr == 1)
    visibility.append(True)  
    workingday_buttons.append(dict(
        label=wd_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count (Spring, {wd_label}, Clear)"}]
    ))

#Dropdown 4: Weather
weather_buttons = []
for wthr, wthr_label in weather_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for w in weather_labels:
                    visibility.append(w == wthr and s == 1 and y_col == "count" and wd == 1)
    visibility.append(True)  
    weather_buttons.append(dict(
        label=wthr_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Temperature and Count (Spring, Weekday, {wthr_label})"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.55,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.65,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=workingday_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=1
        ),
        dict(
            buttons=weather_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        )
    ],
    title="Temperature and Count (Spring, Weekday, Clear)",
    height=600
)

fig.show()

In [ ]:
#Dictionary for dropdown menu labels
season_labels = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}
y_options = ["count", "registered", "casual"]
workingday_labels = {0: "Weekend", 1: "Weekday"}
#weather_labels = {
    #1: "Clear, Few clouds,<br>Partly Cloudy", 
    #2: "Mist + Cloudy,<br>Mist + Few clouds,<br>Mist", 
    #3: "Light Snow,<br>Light Rain + Thunderstorm +<br>Scattered Clouds,<br>Light Rain + Scattered Clouds", 
    #4: "Heavy Rain + Ice Pellets +<br>Thunderstorm + Mist,<br>Snow + Fog"}

weather_labels = {
    1: "Clear",
    2: "Mist",
    3: "Light Precip",
    4: "Heavy Precip"
}

fig = go.Figure()

global_sizeref = 2. * df_eda["count"].max() / 40**2

for y_col in y_options:
    for season, label in season_labels.items():
        for wd, wd_label in workingday_labels.items():
            for wthr, wthr_label in weather_labels.items():
                df_season = df_eda[
                    (df_eda["season"] == season) &
                    (df_eda["workingday"] == wd) &
                    (df_eda["weather"] == wthr)
                ]
                
                if df_season.empty:
                    continue 
                
                fig.add_trace(
                    go.Scatter(
                        x=df_season["humidity"],
                        y=df_season[y_col],
                        mode="markers",
                        marker=dict(
                            size=df_season["count"],
                            color=df_season["humidity"],
                            colorscale="blues",
                            showscale=False,
                            sizemode="area",
                            sizeref=global_sizeref, 
                            sizemin=4
                        ),
                        name=f"{label} - {y_col} - {wd_label} - {wthr_label}",
                        visible=True if (season==1 and y_col=="count" and wd==1 and wthr==1) else False,
                    )
                )

fig.add_trace(
    go.Scatter(
        x=[None],
        y=[None],
        mode="markers",
        marker=dict(
            colorscale="blues",
            cmin=raw_df["humidity"].min(),
            cmax=raw_df["humidity"].max(),
            colorbar=dict(
                title="Humidity",
                thickness=20,
                len=0.8,
                x=1.05,
                y=0.5,
                yanchor="middle"
            ),
            showscale=True
        ),
        hoverinfo="none",
        showlegend=False
    )
)

fig.update_xaxes(title="Humidity")
fig.update_yaxes(title="Count")

#Dropdown 1: Season
season_buttons = []
for season, label in season_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(s == season and y_col == "count" and wd == 1 and wthr == 1)
    visibility.append(True)  
    season_buttons.append(dict(
        label=label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count ({label}, Weekday, Clear)"}]
    ))

#Dropdown 2: Customer type
y_buttons = []
for y_col in y_options:
    visibility = []
    for y_opt in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(y_opt == y_col and s == 1 and wd == 1 and wthr == 1)
    visibility.append(True)  
    y_buttons.append(dict(
        label=y_col.capitalize(),
        method="update",
        args=[{"visible": visibility},
              {"yaxis": {"title": y_col.capitalize()},
               "title": f"Humidity and {y_col.capitalize()} (Spring, Weekday, Clear)"}]
    ))

#Dropdown 3: Working day
workingday_buttons = []
for wd, wd_label in workingday_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for w in workingday_labels:
                for wthr in weather_labels:
                    visibility.append(w == wd and s == 1 and y_col == "count" and wthr == 1)
    visibility.append(True)  
    workingday_buttons.append(dict(
        label=wd_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count (Spring, {wd_label}, Clear)"}]
    ))

#Dropdown 4: Weather
weather_buttons = []
for wthr, wthr_label in weather_labels.items():
    visibility = []
    for y_col in y_options:
        for s in season_labels:
            for wd in workingday_labels:
                for w in weather_labels:
                    visibility.append(w == wthr and s == 1 and y_col == "count" and wd == 1)
    visibility.append(True)  
    weather_buttons.append(dict(
        label=wthr_label,
        method="update",
        args=[{"visible": visibility},
              {"title": f"Humidity and Count (Spring, Weekday, {wthr_label})"}]
    ))

fig.update_layout(
    updatemenus=[
        dict(
            buttons=season_buttons,
            direction="down",
            showactive=True,
            x=0.55,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=y_buttons,
            direction="down",
            showactive=True,
            x=0.65,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        ),
        dict(
            buttons=workingday_buttons,
            direction="down",
            showactive=True,
            x=0.75,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=1
        ),
        dict(
            buttons=weather_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="left",
            y=1.2,
            yanchor="top",
            pad={"r":5, "t":5},
            font={"size":13},
            active=0
        )
    ],
    title="Humidity and Count (Spring, Weekday, Clear)",
    height=600
)

fig.show()